# Training of Generative Models on the Australian Credit Approval Dataset

## Introduction

This notebook marks the beginning of the training phase on the Australian Credit Approval dataset, extending the methodology previously applied to the German Credit Risk dataset. As before, we focus on four generative models—CopulaGAN, TVAE, CTGAN, and GReaT (with the distil-GPT2 backbone)—which are trained to learn the statistical properties of the data and produce synthetic samples.

Since the overall workflow remains consistent with that described in the first training notebook, we will avoid repeating technical details already introduced there, and instead emphasize aspects specific to the Australian dataset. The synthetic data produced here will later serve as the foundation for comparative evaluation alongside the models trained on the German dataset.

## 0 Imports and function definition

We start by importing the necessary libraries for data preprocessing, model training, and persistence. These include tools for handling tabular data, applying preprocessing pipelines, training the selected generative models, and saving the resulting objects for later use.

In [ ]:
!pip install be_great peft==0.9.0 sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import os
from be_great import GReaT
from sdv.single_table import TVAESynthesizer, CTGANSynthesizer, CopulaGANSynthesizer
from sdv.metadata import SingleTableMetadata
import pickle

In [ ]:
os.environ["WANDB_DISABLED"] = "true" #to avoid the requests from WANDB

The preprocessing function preprocess_dataset is the same as defined in the first notebook and is used here without modification to standardize data types and handle missing values.

In [ ]:
def preprocess_dataset(df, target_column, categorical_features, numerical_features):
    df = df.copy()

    for col in categorical_features:
        df[col] = df[col].astype('object')

    for col in numerical_features:
        df[col] = df[col].astype('float')

    for col in categorical_features:
        df[col] = df[col].fillna('Unknown')

    for col in numerical_features:
        df[col] = df[col].fillna(df[col].mean())

    return df


The training functions for CopulaGAN, TVAE, CTGAN, and GReaT are the same as those defined in the previous notebooks. Here, they are applied to the Australian Credit Approval dataset using the corresponding file paths for model serialization.

In [ ]:
def train_and_save_model_CopulaGAN(df):

  filename = "copulagan_australian.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = CopulaGANSynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
      pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_TVAE(df):

  filename = "tvae_australian.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = TVAESynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
      pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_CTGAN(df):

  filename = "ctgan_australian.pkl"

  metadata = SingleTableMetadata()
  metadata.detect_from_dataframe(data=df)
  synthesizer = CTGANSynthesizer(metadata)
  synthesizer.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(synthesizer, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def train_and_save_model_distil_GReaT(df, batch_size, epochs, efficient_finetuning):

  filename = "distil_GReaT_australian.pkl"

  model = GReaT(llm="distilgpt2", batch_size=batch_size, epochs=epochs, fp16=True, efficient_finetuning=efficient_finetuning)
  model.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

## 1 Dataset loading

The Australian Credit Approval dataset is first loaded from its CSV file into a pandas DataFrame. This structured format allows for efficient preprocessing and subsequent model training. The dataset can be accessed from the following source: https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval

In [ ]:
#Load Australian Credit Approval
australian_df = pd.read_csv('australian_credit_approval.csv')


Based on the dataset documentation, we identify the categorical and numerical features of the Australian Credit Approval dataset. The dataset is then preprocessed using the preprocess_dataset function, which standardizes data types and imputes missing values, producing a clean dataset ready for training the generative models.

In [ ]:
australian_categorical_features = ['A1', 'A4', 'A5', 'A6', 'A8', 'A9', 'A11', 'A12']
australian_numerical_features = ['A2', 'A3', 'A7', 'A10', 'A13', 'A14']

In [ ]:
preprocessed_dataset = preprocess_dataset(australian_df, 'Class', australian_categorical_features, australian_numerical_features)

/tmp/ipython-input-4-1313495981.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna('Unknown')


## 2 Train

Following preprocessing, we proceed to train the generative models using the previously defined training functions. This ensures that each model learns the structure of the Australian Credit Approval dataset and is saved for subsequent synthetic data generation.

### CopulaGAN

In [ ]:
train_and_save_model_CopulaGAN(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### TVAE

In [ ]:
train_and_save_model_TVAE(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### CTGAN

In [ ]:
train_and_save_model_CTGAN(preprocessed_dataset)

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


### Distil-GReaT

In [ ]:
train_and_save_model_distil_GReaT(preprocessed_dataset, 32, 200, None)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/be_great/great.py:174: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `GReaTTrainer.__init__`. Use `processing_class` instead.
  great_trainer = GReaTTrainer(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,0.862600
1000,0.718800
1500,0.700200
2000,0.689200
2500,0.680100
3000,0.673300
3500,0.667500
4000,0.663000
